# Milestone 3: Evaluation, Plots, and Results Compilation

This notebook loads the experiments matrix results from `results/metrics_all.csv`, generates the primary visualizations for our study, and performs comparisons between controllers, fixed baselines, and the oracle ceiling.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

from src.bench.harness import run_experiment_stream
from src.run_experiments import get_controller_fn

os.makedirs("results/figures", exist_ok=True)
print("Libraries imported and directories verified.")

## 1. Loading and Inspecting the Metrics
We load the global metrics compiled across all workloads, policies, and batch sizes.

In [ ]:
df = pd.read_csv("results/metrics_all.csv")
print(f"Loaded {len(df)} experiment runs.")
df.head(15)

## 2. Speedup-vs-Policy Comparison (Batch Size = 1)
We compare each adaptive controller against the best static fixed length baseline and the oracle ceiling for each workload.

In [ ]:
df_bs1 = df[df["batch_size"] == 1]
workloads = ["humaneval", "gsm8k", "mt_bench", "spec_bench", "mixed"]
policies = ["entropy_threshold", "epsilon_greedy", "ucb", "history", "oracle"]

# Find best fixed length per workload first
best_fixed_speedups = {}
for w in workloads:
    fixed_w = df_bs1[(df_bs1["workload"] == w) & (df_bs1["policy"].str.startswith("fixed_"))]
    best_fixed_speedups[w] = fixed_w["net_speedup"].max()

print("=== Best Fixed Speedups ===")
for w, val in best_fixed_speedups.items():
    print(f"{w:<12}: {val:.3f}x")

# Plotting speedup bars
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(workloads))
width = 0.15

for idx, p in enumerate(policies):
    p_df = df_bs1[df_bs1["policy"] == p]
    speedups = [p_df[p_df["workload"] == w]["net_speedup"].values[0] for w in workloads]
    ax.bar(x + (idx - 2) * width, speedups, width, label=p)

# Draw horizontal lines or reference points for best fixed
for idx, w in enumerate(workloads):
    ax.plot([idx - 0.4, idx + 0.4], [best_fixed_speedups[w], best_fixed_speedups[w]], 
             color='black', linestyle='--', alpha=0.7, label='Best Fixed' if idx == 0 else "")

ax.set_title("Speculative Decoding Speedup comparison (Batch Size = 1)")
ax.set_xticks(x)
ax.set_xticklabels(workloads)
ax.set_ylabel("Net Speedup (x)")
ax.legend()
ax.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.savefig("results/figures/fig4_speedup_comparison.png", dpi=300)
plt.close()
print("Speedup comparison plot saved to results/figures/fig4_speedup_comparison.png")

## 3. Wasted Token Comparison
We analyze the efficiency of each policy by plotting the wasted-draft-tokens ratio (wasted-draft-tokens per accepted token).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(workloads))
width = 0.15

for idx, p in enumerate(policies):
    p_df = df_bs1[df_bs1["policy"] == p]
    wasted = [p_df[p_df["workload"] == w]["wasted_tokens_per_accepted"].values[0] for w in workloads]
    ax.bar(x + (idx - 2) * width, wasted, width, label=p)

ax.set_title("Wasted Draft Tokens per Accepted Token (Lower is Better)")
ax.set_xticks(x)
ax.set_xticklabels(workloads)
ax.set_ylabel("Wasted Tokens Ratio")
ax.legend()
ax.grid(True, axis='y', linestyle='--', alpha=0.6)
plt.savefig("results/figures/fig5_wasted_tokens.png", dpi=300)
plt.close()
print("Wasted tokens plot saved to results/figures/fig5_wasted_tokens.png")

## 4. Online Bandit Convergence and Regret Analysis
We trace the cumulative speedup of UCB and EpsilonGreedy from a cold start on the `mixed` workload to show how quickly they learn.

In [ ]:
print("=== Tracing Bandit Convergence on Mixed Traffic ===")
steps_to_trace = 1000
ucb_records = run_experiment_stream("mixed", "ucb", get_controller_fn("ucb"), num_steps=steps_to_trace, batch_size=1, seed=42)
eps_records = run_experiment_stream("mixed", "epsilon_greedy", get_controller_fn("epsilon_greedy"), num_steps=steps_to_trace, batch_size=1, seed=42)
oracle_records = run_experiment_stream("mixed", "oracle", get_controller_fn("oracle"), num_steps=steps_to_trace, batch_size=1, seed=42)

ucb_cum_speedup = np.cumsum([r["net_speedup"] for r in ucb_records]) / (np.arange(steps_to_trace) + 1)
eps_cum_speedup = np.cumsum([r["net_speedup"] for r in eps_records]) / (np.arange(steps_to_trace) + 1)
oracle_cum_speedup = np.cumsum([r["net_speedup"] for r in oracle_records]) / (np.arange(steps_to_trace) + 1)

plt.figure(figsize=(10, 5))
plt.plot(oracle_cum_speedup, label='Oracle Ceiling', color='black', linestyle='--')
plt.plot(ucb_cum_speedup, label='UCB Bandit (c=0.5)', color='royalblue', linewidth=2)
plt.plot(eps_cum_speedup, label='Epsilon-Greedy (eps=0.1)', color='orange', linewidth=2)
plt.title("Bandit Convergence: Cumulative Average Speedup over Time")
plt.xlabel("Generation Step")
plt.ylabel("Cumulative Average Speedup (x)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.savefig("results/figures/fig6_bandit_convergence.png", dpi=300)
plt.close()
print("Convergence plot saved to results/figures/fig6_bandit_convergence.png")

## 5. Batch Size Sweep Curve (Batch Interference)
We plot the speedup of different policies as the batch size increases ($B \in \{1, 8, 32, 64\}$) on `mixed` traffic to evaluate batch interference.

In [ ]:
df_mixed = df[df["workload"] == "mixed"]
batch_sizes = [1, 8, 32, 64]

plt.figure(figsize=(10, 6))
plot_policies = ["fixed_3", "entropy_threshold", "ucb", "history", "oracle"]
colors = {"fixed_3": "grey", "entropy_threshold": "teal", "ucb": "royalblue", "history": "orange", "oracle": "black"}
styles = {"fixed_3": "-.", "entropy_threshold": "-", "ucb": "-", "history": "-", "oracle": "--"}

for p in plot_policies:
    p_df = df_mixed[df_mixed["policy"] == p].sort_values("batch_size")
    plt.plot(p_df["batch_size"], p_df["net_speedup"], marker='o', label=p, 
             color=colors[p], linestyle=styles[p], linewidth=2)

plt.title("Speculative Decoding Speedup vs. Serving Batch Size")
plt.xlabel("Batch Size (B)")
plt.ylabel("Net Speedup (x)")
plt.xscale('log', base=2)
plt.xticks(batch_sizes, [str(b) for b in batch_sizes])
plt.grid(True, which="both", linestyle='--', alpha=0.5)
plt.legend()
plt.savefig("results/figures/fig7_batch_sweep.png", dpi=300)
plt.close()
print("Batch sweep plot saved to results/figures/fig7_batch_sweep.png")